# Processing Haid et al. (2023) ocean data

Created by Yanmei Tian <yanmeiti@buffalo.edu>

This notebook contains the core processing used to generate the Haid et al. (2023) ISMIP7 ocean-forcing products.


- **cold**: last 20 years (1998–2017) of the reference experiment
- **warm**: last 20 years (1998–2017) of perturbation experiment SA_G
- SA_S and SA_W are not processed
- the final available spin-up cycle (`cycle4`) is used for both cases

Final variables are named `thetao`, `so`, and `tf`.

### 1. Imports and paths

Run this notebook with the `ismip7_dev` environment. The external programs `i7aof_extrap_horizontal` and `i7aof_extrap_vertical` must be available.

In [ ]:
from pathlib import Path
import os
import subprocess

import numpy as np
import xarray as xr
from jinja2 import Environment, FileSystemLoader, StrictUndefined

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

HERE = Path.cwd().resolve()
if not (HERE / "namelisttemplate.nml").is_file():
    HERE = Path("/home/yanmeiti/AIS_Ocean/parameterisations/process_data")

PROJECT_DIR = HERE.parents[1]
DATA_DIR = PROJECT_DIR / "Data"
INPUT_DIR = DATA_DIR / "ocean_climatology" / "Haid"
TOPO_DIR = DATA_DIR / "topg"
OUTPUT_DIR = DATA_DIR / "output"
WORK_DIR = INPUT_DIR / "processing_work"
TEMPLATE = HERE / "namelisttemplate.nml"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEDMAP_FILE = TOPO_DIR / "bedmap3_ismip_8km.nc"
GRID_FILE = TOPO_DIR / "ismip_8km_60m_grid.nc"
BASIN_FILE = TOPO_DIR / "basin_numbers_ismip8km_v2.nc"

# Set True only when intentionally rerunning existing products.
OVERWRITE = False

In [ ]:
EXPERIMENTS = {
    "cold": INPUT_DIR / "Haid et al. (2023) reference experiment" /
            "FEZOM.REF.cycle4.1979-2017.TS.aym.ismip.91levels.nc",
    "warm": INPUT_DIR / "Haid et al. (2023) perturbation experiment SA_G" /
            "FEZOM.SA_G.cycle4.1979-2017.TS.aym.ismip.91levels.nc",
}

for mode, source_file in EXPERIMENTS.items():
    if not source_file.is_file():
        raise FileNotFoundError(source_file)
    print(mode, "->", source_file.name)

### 2. Calculate the 1998–2017 climatologies

rename variables:

```text
x/y/z/t       → x/y/z/time
temp/salt     → theta_ocean/salinity_ocean
```

The 91 levels are interpolated to the 90-level `z_extrap` grid required by the extrapolation tools. Values beneath floating and grounded ice are removed.

In [ ]:
def check_output(path):
    path = Path(path)
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f"{path} already exists; set OVERWRITE=True to replace it")
    path.parent.mkdir(parents=True, exist_ok=True)


def make_climatology(mode, source_file):
    output_file = WORK_DIR / f"Haid2023_{mode}_prepared.nc"
    check_output(output_file)

    # time chunks avoid loading all 39 annual 3-D fields simultaneously
    with xr.open_dataset(source_file, chunks={"anzyears": 1}) as source:
        ds = source.swap_dims({
            "anzxismip": "x",
            "anzyismip": "y",
            "anzzismip": "z",
            "anzyears": "t",
        })
        ds = ds.rename({
            "t": "time",
            "temp": "theta_ocean",
            "salt": "salinity_ocean",
        })[["theta_ocean", "salinity_ocean"]]

        # Select exactly 1998–2017, including both endpoints.
        selected = ds.sel(time=slice(1998, 2017))
        expected_years = np.arange(1998, 2018, dtype=float)
        np.testing.assert_allclose(selected.time.values, expected_years)

        climatology = selected.mean("time", keep_attrs=True)

        # The legacy Fortran tools require a length-one CF time dimension.
        climatology = climatology.expand_dims(time=[0.0])
        climatology.time.attrs = {
            "units": "days since 2000-01-01 00:00:00",
            "calendar": "proleptic_gregorian",
            "standard_name": "time",
            "long_name": "dummy time for the 1998-2017 climatology",
        }

        # Native z: 0, -20, ..., -1800 m
        # z_extrap: -10, -30, ..., -1790 m
        with xr.open_dataset(GRID_FILE) as grid:
            climatology = climatology.interp(
                z=grid["z_extrap"], method="linear"
            )

        # Retain open-ocean values; cavities and grounded ice are extrapolated.
        with xr.open_dataset(BEDMAP_FILE) as bedmap:
            open_ocean = (
                (bedmap["floating_frac"] == 0) &
                (bedmap["grounded_frac"] == 0)
            )
            climatology["theta_ocean"] = climatology.theta_ocean.where(open_ocean)
            climatology["salinity_ocean"] = climatology.salinity_ocean.where(open_ocean)

        climatology = climatology.transpose("time", "z_extrap", "y", "x")
        climatology.attrs.update({
            "title": f"Haid et al. (2023) {mode} ocean climatology",
            "source_file": str(source_file),
            "history": "1998-2017 mean; interpolated to z_extrap; ice masked",
        })

        encoding = {
            "theta_ocean": {"_FillValue": -9999.0, "dtype": "float32"},
            "salinity_ocean": {"_FillValue": -9999.0, "dtype": "float32"},
            "time": {"_FillValue": None},
        }
        climatology.to_netcdf(
            output_file,
            encoding=encoding,
            unlimited_dims="time",
        )

    print("Wrote", output_file)
    return output_file

In [ ]:
# Run once to create the cold and warm climatologies.
# Skip this cell when the prepared files already exist.
# Set RUN_CLIMATOLOGY=True to run.
RUN_CLIMATOLOGY = False

if RUN_CLIMATOLOGY:
    for mode, source_file in EXPERIMENTS.items():
        make_climatology(mode, source_file)
else:
    print("Climatology calculation skipped; set RUN_CLIMATOLOGY=True to run.")

### 4. Create namelists and run horizontal/vertical extrapolation

Temperature and salinity are extrapolated independently. 
Horizontal extrapolation first, vertical extrapolation second.

In [ ]:
FIELD_VARIABLES = {
    "T": "theta_ocean",
    "S": "salinity_ocean",
}


def make_namelist(mode, suffix):
    prepared_file = WORK_DIR / f"Haid2023_{mode}_prepared.nc"
    horizontal_file = WORK_DIR / f"Haid2023_{mode}_{suffix}_horizontal.nc"
    extrapolated_file = WORK_DIR / f"Haid2023_{mode}_{suffix}_zextrap.nc"
    namelist_file = WORK_DIR / f"namelist_haid_{mode}_{suffix}.nml"

    if not prepared_file.is_file():
        raise FileNotFoundError(prepared_file)
    check_output(horizontal_file)
    check_output(extrapolated_file)

    environment = Environment(
        loader=FileSystemLoader(str(TEMPLATE.parent)),
        undefined=StrictUndefined,
        keep_trailing_newline=True,
    )
    text = environment.get_template(TEMPLATE.name).render(
        file_in=str(prepared_file.resolve()),
        file_out_horizontal=str(horizontal_file.resolve()),
        file_out=str(extrapolated_file.resolve()),
        file_basin=str(BASIN_FILE.resolve()),
        file_topo=str(BEDMAP_FILE.resolve()),
        variable=FIELD_VARIABLES[suffix],
        z_name="z_extrap",
    )
    namelist_file.write_text(text.replace("\r\n", "\n"), encoding="utf-8")
    return namelist_file


def run_extrapolation():
    namelists = [
        make_namelist(mode, suffix)
        for mode in ("cold", "warm")
        for suffix in ("T", "S")
    ]

    for namelist in namelists:
        subprocess.run(
            ["i7aof_extrap_horizontal", str(namelist)],
            check=True,
        )

    for namelist in namelists:
        subprocess.run(
            ["i7aof_extrap_vertical", str(namelist)],
            check=True,
        )

In [ ]:
# set RUN_EXTRAPOLATION=True to run.
RUN_EXTRAPOLATION = False

if RUN_EXTRAPOLATION:
    run_extrapolation()
else:
    print("Extrapolation skipped; set RUN_EXTRAPOLATION=True to run.")

### 5. Interpolate to the final grid and rename variables

The extrapolated 90-level fields are interpolated to the final 30-level grid. The dummy time dimension is removed, and final temperature/salinity variables are written as `thetao` and `so`.

In [ ]:
FINAL_NAMES = {
    "T": ("theta_ocean", "thetao", "deg C", "sea water potential temperature"),
    "S": ("salinity_ocean", "so", "psu", "sea water salinity"),
}


def finalize_field(mode, suffix):
    input_file = WORK_DIR / f"Haid2023_{mode}_{suffix}_zextrap.nc"
    output_file = OUTPUT_DIR / f"Haid2023_{mode}_{suffix}.nc"
    check_output(output_file)

    source_name, final_name, units, long_name = FINAL_NAMES[suffix]

    with xr.open_dataset(input_file) as ds, xr.open_dataset(GRID_FILE) as grid:
        result = ds[[source_name]].interp(
            z_extrap=grid["z"], method="linear"
        )
        result = result.squeeze("time", drop=True)
        result = result.drop_vars(["lat", "lon", "z_extrap"], errors="ignore")
        result = result.rename({source_name: final_name})
        result.encoding.pop("unlimited_dims", None)
        result[final_name].attrs = {
            "long_name": long_name,
            "units": units,
        }
        result.attrs.update({
            "title": f"Haid et al. (2023) {mode} ocean forcing",
            "history": "horizontal and vertical extrapolation; final 30-level z grid",
        })
        result.to_netcdf(
            output_file,
            encoding={final_name: {"_FillValue": -9999.0, "dtype": "float32"}},
        )

    print("Wrote", output_file)
    return output_file

### 6. Calculate thermal forcing



In [ ]:
def calculate_thermal_forcing(mode):
    temperature_file = OUTPUT_DIR / f"Haid2023_{mode}_T.nc"
    salinity_file = OUTPUT_DIR / f"Haid2023_{mode}_S.nc"
    output_file = OUTPUT_DIR / f"Haid2023_{mode}_TF.nc"
    check_output(output_file)

    with xr.open_dataset(temperature_file) as temperature, \
         xr.open_dataset(salinity_file) as salinity:
        pressure = 1028.0 * 9.81 * temperature.z * -1.0
        freezing_point = (
            -0.0572 * salinity.so + 0.0788 - 7.77e-8 * pressure
        )
        result = xr.Dataset({
            "tf": (temperature.thetao - freezing_point).astype("float32")
        })
        result.tf.attrs = {
            "long_name": "ocean thermal forcing",
            "units": "deg C",
        }
        result.attrs.update({
            "title": f"Haid et al. (2023) {mode} thermal forcing",
            "history": "linearized freezing point following Reese et al. (2018)",
        })
        result.to_netcdf(
            output_file,
            encoding={"tf": {"_FillValue": -9999.0, "dtype": "float32"}},
        )

    print("Wrote", output_file)
    return output_file

In [ ]:
# Set RUN_FINAL_FILES=True to run.
RUN_FINAL_FILES = False

if RUN_FINAL_FILES:
    for mode in ("cold", "warm"):
        finalize_field(mode, "T")
        finalize_field(mode, "S")
        calculate_thermal_forcing(mode)
else:
    print("Final-file generation skipped; set RUN_FINAL_FILES=True to run.")

### Final products


```text
Haid2023_cold_T.nc   (thetao)
Haid2023_cold_S.nc   (so)
Haid2023_cold_TF.nc  (tf)
Haid2023_warm_T.nc   (thetao)
Haid2023_warm_S.nc   (so)
Haid2023_warm_TF.nc  (tf)
```
